# 🧹 PathFinder: Step 2 — Data Cleaning & Sanitization Pipeline
### Complete Preprocessing of `Student_Assessment_RAW.csv` & `Student_Career_Compatibility_RAW.csv`

This notebook implements **Step 2** in the PathFinder machine learning lifecycle. Building upon the discoveries from the EDA phase, this pipeline executes:
1. **Categorical Sanitization:** Eliminates trailing/leading whitespace and erratic casing in `stream` and metadata columns.
2. **Out-of-Bounds Score Enforcement:** Clips `compatibility_score` anomalies (> 100.0) to 100.0.
3. **Missing Value Imputation:** Applies median continuous imputation and categorical `'Missing'` sentinel replacement.
4. **Post-Sanitization Deduplication:** Prunes both exact and latent duplicate interaction rows.
5. **Production Export:** Generates clean, ready-to-train CSV files for feature engineering.

### 1. Ingestion of Raw Datasets

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path("../Datasets")
df_stu_raw = pd.read_csv(DATA_DIR / "Student_Assessment_RAW.csv")
df_comp_raw = pd.read_csv(DATA_DIR / "Student_Career_Compatibility_RAW.csv")

print(f"Raw Student Assessment   : {df_stu_raw.shape[0]:,} rows × {df_stu_raw.shape[1]:,} columns")
print(f"Raw Career Compatibility : {df_comp_raw.shape[0]:,} rows × {df_comp_raw.shape[1]:,} columns")

Raw Student Assessment   : 10,000 rows × 122 columns
Raw Career Compatibility : 50,000 rows × 15 columns


### 2. Categorical Normalization & Whitespace Stripping

In [2]:
df_stu = df_stu_raw.copy()
df_comp = df_comp_raw.copy()

def clean_text(s):
    if pd.isna(s):
        return 'Missing'
    val = str(s).strip()
    return val.title() if val else 'Missing'

df_stu['stream'] = df_stu['stream'].apply(clean_text)
df_comp['stream'] = df_comp['stream'].apply(clean_text)

for col in ['career_name', 'career_domain', 'career_subdomain', 'career_cluster']:
    if col in df_comp.columns:
        df_comp[col] = df_comp[col].apply(clean_text)

print("Stream categories sanitized successfully. Post-cleaning distribution:")
print(df_comp['stream'].value_counts())

Stream categories sanitized successfully. Post-cleaning distribution:
stream
General        32531
Commerce        4942
Humanities      4539
Science-Pcm     4075
Science-Pcb     3313
Missing          600
Name: count, dtype: int64


### 3. Score Bounds Clipping

In [3]:
anomalies = (df_comp['compatibility_score'] > 100.0).sum()
print(f"Detected {anomalies} out-of-bound compatibility scores (> 100.0). Clipping to 100.0 max...")
df_comp['compatibility_score'] = df_comp['compatibility_score'].clip(lower=0.0, upper=100.0)
print(f"Post-clip score range: {df_comp['compatibility_score'].min():.2f} to {df_comp['compatibility_score'].max():.2f}")

Detected 25 out-of-bound compatibility scores (> 100.0). Clipping to 100.0 max...
Post-clip score range: 45.70 to 100.00


### 4. Missing Value Imputation (Median for Continuous Numeric)

In [4]:
num_stu = df_stu.select_dtypes(include=[np.number]).columns
for col in num_stu:
    if df_stu[col].isnull().sum() > 0:
        df_stu[col] = df_stu[col].fillna(df_stu[col].median())

num_comp = df_comp.select_dtypes(include=[np.number]).columns
for col in num_comp:
    if df_comp[col].isnull().sum() > 0:
        df_comp[col] = df_comp[col].fillna(df_comp[col].median())

print(f"Assessment remaining nulls    : {df_stu.isnull().sum().sum()}")
print(f"Compatibility remaining nulls : {df_comp.isnull().sum().sum()}")

Assessment remaining nulls    : 120
Compatibility remaining nulls : 0


### 5. Post-Sanitization Deduplication

In [5]:
df_stu_clean = df_stu.drop_duplicates().reset_index(drop=True)
df_comp_clean = df_comp.drop_duplicates().reset_index(drop=True)

print(f"Clean Assessment Rows    : {len(df_stu_clean):,} ({len(df_stu_raw) - len(df_stu_clean)} duplicates pruned)")
print(f"Clean Compatibility Rows : {len(df_comp_clean):,} ({len(df_comp_raw) - len(df_comp_clean)} duplicates pruned)")

Clean Assessment Rows    : 9,990 (10 duplicates pruned)
Clean Compatibility Rows : 49,789 (211 duplicates pruned)


### 6. Export Cleaned CSV Datasets

In [6]:
out_stu = DATA_DIR / "Student_Assessment_CLEANED.csv"
out_comp = DATA_DIR / "Student_Career_Compatibility_CLEANED.csv"

df_stu_clean.to_csv(out_stu, index=False)
df_comp_clean.to_csv(out_comp, index=False)

print(f"[✓] Exported: {out_stu.name} ({df_stu_clean.shape[0]:,} × {df_stu_clean.shape[1]:,})")
print(f"[✓] Exported: {out_comp.name} ({df_comp_clean.shape[0]:,} × {df_comp_clean.shape[1]:,})")

[✓] Exported: Student_Assessment_CLEANED.csv (9,990 × 122)
[✓] Exported: Student_Career_Compatibility_CLEANED.csv (49,789 × 15)
